# Programmatic access to postsynaptic target prediction data

In this notebook, we'll see how to use [CAVEclient](https://www.caveconnecto.me/CAVEclient/) to access the annotations generated by the postsynaptic target classifier described in the paper [Pedigo et al. 2026 - "A quantitative census of millions of postsynaptic structures in a large electron microscopy volume of mouse visual cortex"](https://www.biorxiv.org/content/10.64898/2026.02.19.706834).

Prerequisites: 
- make sure you have [installed CAVEclient](https://www.caveconnecto.me/CAVEclient/installation/) (typically just `pip install caveclient` or `uv add caveclient`).
- make sure you have [authenticated](https://www.caveconnecto.me/CAVEclient/tutorials/authentication/) for the `microns65_public` datastack. You should only have to do this once per machine. 

## Initialize a CAVEclient object

Setting the version here is important - the underlying segmentation is changing over time, so while these annotations are tracked and will still be there, they will potentially correspond to different root IDs. Much more in depth explanation is available in the [MICrONS tutorials](https://tutorial.microns-explorer.org/materialization-version.html).

In [1]:
from caveclient import CAVEclient

version = 1718
client = CAVEclient("minnie65_public", version=version)

## Query annotations for a cell's input synapses

In [2]:
# the root ID of a neuron we are interested in
# note that root IDs are updated as the segmentation changes, see the MICrONS tutorials
root_id = 864691135276234469

# this single query will pull both the synapse information and information about the
# predicted labels from the postsynaptic target classifier
annotated_post_synapses = client.materialize.tables.synapse_target_predictions_ssa_v2(
    post_pt_root_id=root_id
).query(
    desired_resolution=[1, 1, 1],  # specify this to get data back in nm units
    split_positions=True,  # unpacks the synapse center point into separate x, y, z columns
)

Table Owner Notice on synapse_target_predictions_ssa_v2: Table in development, contact Ben Pedigo with questions or feedback. Citation  forthcoming, reach out if wanting to use for publication., Using non-pandas query execution is deprecated as it can mangle types, please upgrade caveclient to >=8.0.0 to use pandas for improved type handling.


We can have a look at some of the more important columns in the return:

In [3]:
annotated_post_synapses[
    [
        "id",  # the ID of the synapse
        "size",  # the size of the synapse in voxels
        "pre_pt_root_id",  # the root ID of the presynaptic partner
        "post_pt_root_id",  # the root ID of the postsynaptic partner
        "tag",  # the predicted classifier label
        "value",  # the confidence of the prediction
        "ctr_pt_position_x",  # the center of the synapse x coordinate, in nm
        "ctr_pt_position_y",  # the center of the synapse y coordinate, in nm
        "ctr_pt_position_z",  # the center of the synapse z coordinate, in nm
    ]
]

,id,size,pre_pt_root_id,post_pt_root_id,tag,value,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z
0,174384766,5296,864691135342593477,864691135276234469,shaft,0.975994,750168.0,546016.0,840320.0
1,199977114,5440,864691134269898111,864691135276234469,shaft,0.551134,802520.0,394208.0,754160.0
2,171643719,4780,864691136775802222,864691135276234469,shaft,0.914188,738312.0,536516.0,859040.0
3,163925645,2444,864691135029751054,864691135276234469,spine,0.999912,728264.0,549936.0,798120.0
4,158110151,9184,864691134111280096,864691135276234469,spine,0.997760,706568.0,382776.0,878360.0
...,...,...,...,...,...,...,...,...,...
3071,202016285,1700,864691135572327518,864691135276234469,shaft,0.965703,800856.0,554984.0,760520.0
3072,193640878,6764,864691136465038299,864691135276234469,spine,0.999893,786568.0,408880.0,775240.0
3073,202016309,1336,864691136006585180,864691135276234469,spine,0.999994,800904.0,554312.0,757920.0
3074,165403164,2568,864691136136326283,864691135276234469,shaft,0.989726,726136.0,498240.0,870680.0


## Query annotations for a cell's output synapses

Simply changing the filter to `pre_pt_root_id` will now query for output synapses from this cell.

In [4]:
annotated_pre_synapses = client.materialize.tables.synapse_target_predictions_ssa_v2(
    pre_pt_root_id=root_id
).query(
    desired_resolution=[1, 1, 1],  # specify this to get data back in nm units
    split_positions=True,  # unpacks the synapse center point into separate x, y, z columns
)

Table Owner Notice on synapse_target_predictions_ssa_v2: Table in development, contact Ben Pedigo with questions or feedback. Citation  forthcoming, reach out if wanting to use for publication., Using non-pandas query execution is deprecated as it can mangle types, please upgrade caveclient to >=8.0.0 to use pandas for improved type handling.


In [5]:
annotated_pre_synapses[
    [
        "id",  # the ID of the synapse
        "size",  # the size of the synapse in voxels
        "pre_pt_root_id",  # the root ID of the presynaptic partner
        "post_pt_root_id",  # the root ID of the postsynaptic partner
        "tag",  # the predicted classifier label
        "value",  # the confidence of the prediction
        "ctr_pt_position_x",  # the center of the synapse x coordinate, in nm
        "ctr_pt_position_y",  # the center of the synapse y coordinate, in nm
        "ctr_pt_position_z",  # the center of the synapse z coordinate, in nm
    ]
]

,id,size,pre_pt_root_id,post_pt_root_id,tag,value,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z
0,280785376,4200,864691135276234469,864691135645366255,spine,0.999923,991376.0,836904.0,1020800.0
1,350708318,1092,864691135276234469,864691135491390823,spine,0.999994,1164840.0,730920.0,722920.0
2,89474100,11164,864691135276234469,864691136991839125,spine,0.999965,554184.0,797960.0,945800.0
3,96966360,4432,864691135276234469,864691135395659765,spine,0.999992,570552.0,754024.0,945680.0
4,241211147,10260,864691135276234469,864691136814622579,spine,0.997649,902768.0,767304.0,1035000.0
...,...,...,...,...,...,...,...,...,...
311,193907175,7120,864691135276234469,864691135698158490,shaft,0.979280,786288.0,741880.0,913240.0
312,267380095,8732,864691135276234469,864691136196042828,spine,0.999993,953224.0,837232.0,1001480.0
313,260166609,6992,864691135276234469,864691136196042828,spine,0.999950,937216.0,835864.0,995840.0
314,189077213,4760,864691135276234469,864691135470301778,spine,0.999812,770816.0,779512.0,840720.0
